## Cell 1 — Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive/ML_final')

import pandas as pd
import numpy as np

!pip install -q dagshub mlflow

import dagshub
dagshub.init(repo_owner='skupr23', repo_name='ml_final', mlflow=True)

import mlflow
print('Tracking URI:', mlflow.get_tracking_uri())

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 820.0 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 53.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 89.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 60.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 64.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=18eb243b-e0a4-402f-88e6-0fdef0cabdf2&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=2b0d697454870848238ccb522d680744edc3b363a35646dc9bfabf3a94fb3b30




Accessing as ggzob23

Initialized MLflow to track repo "skupr23/ml_final"

Repository skupr23/ml_final initialized!

Tracking URI: https://dagshub.com/skupr23/ml_final.mlflow


## Cell 2 — Load the Best Model from the Model Registry

In [2]:
REGISTERED_MODEL_NAME = 'WalmartSalesForecast'
MODEL_STAGE_OR_VERSION = 'latest'

model_uri = f'models:/{REGISTERED_MODEL_NAME}/{MODEL_STAGE_OR_VERSION}'
final_model = mlflow.pyfunc.load_model(model_uri)

print('Loaded model from:', model_uri)
print('Model metadata    :', final_model.metadata.run_id)

2026/07/12 13:14:59 WARNING mlflow.utils.requirements_utils: Detected one or more mismatches between the model's dependencies and the current Python environment:
 - cupy-cuda12x (current: uninstalled, required: cupy-cuda12x==14.0.1)
 - distributed-ucxx-cu12 (current: uninstalled, required: distributed-ucxx-cu12==0.48.0)
To fix the mismatches, call `mlflow.pyfunc.get_model_dependencies(model_uri)` to fetch the model's environment and install dependencies using the resulting environment file.
/usr/local/lib/python3.12/dist-packages/mlflow/sklearn/__init__.py:550: UserWarning: [13:15:01] WARNING: /__w/xgboost/xgboost/src/gbm/gbtree.cc:443: Changing updater from `grow_gpu_hist` to `grow_quantile_histmaker`.
  return cloudpickle.load(f)
/usr/local/lib/python3.12/dist-packages/mlflow/sklearn/__init__.py:550: UserWarning: [13:15:01] WARNING: /__w/xgboost/xgboost/src/context.cc:55: No visible GPU is found, setting device to CPU.
  return cloudpickle.load(f)
/usr/local/lib/python3.12/dist-packa

Loaded model from: models:/WalmartSalesForecast/latest
Model metadata    : b0c239b60ef446489d3c7f853e9f5b5b


## Cell 3 — Load Raw Test Set

In [3]:
test_raw = pd.read_csv('data/test.csv', parse_dates=['Date'])
print('Raw test shape:', test_raw.shape)
test_raw.head()

Raw test shape: (115064, 4)


,Store,Dept,Date,IsHoliday
0,1,1,2012-11-02,False
1,1,1,2012-11-09,False
2,1,1,2012-11-16,False
3,1,1,2012-11-23,True
4,1,1,2012-11-30,False


## Cell 4 — Predict Directly on Raw Test Rows

In [4]:
test_predictions = final_model.predict(test_raw)

print('Predictions generated:', len(test_predictions))
print('Prediction range:', test_predictions.min(), 'to', test_predictions.max())

n_negative = (test_predictions < 0).sum()
print('Negative predictions:', n_negative)

Predictions generated: 115064
Prediction range: -593.3464021668756 to 317849.1149954656
Negative predictions: 979


/usr/local/lib/python3.12/dist-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(


## Cell 5 — Sanity Checks

In [5]:
assert len(test_predictions) == len(test_raw), 'Prediction count mismatch'
assert np.isfinite(test_predictions).all(), 'NaN or infinite predictions found'

if n_negative > 0:
    print(f'Warning: {n_negative} negative predictions - Weekly_Sales cannot be negative in reality.')
    print('Clipping negative predictions to 0 for the submission (does not affect the underlying model).')
    test_predictions = np.clip(test_predictions, 0, None)

Clipping negative predictions to 0 for the submission (does not affect the underlying model).


## Cell 6 — Build Submission CSV

In [6]:
test_ids = (
    test_raw['Store'].astype(str) + '_' +
    test_raw['Dept'].astype(str) + '_' +
    test_raw['Date'].dt.strftime('%Y-%m-%d')
)

submission = pd.DataFrame({
    'Id': test_ids,
    'Weekly_Sales': test_predictions
})

submission.to_csv('submission_final.csv', index=False)
print('Final submission saved to submission_final.csv')
submission.head()

Final submission saved to submission_final.csv


,Id,Weekly_Sales
0,1_1_2012-11-02,36215.830563
1,1_1_2012-11-09,21330.062409
2,1_1_2012-11-16,20736.415244
3,1_1_2012-11-23,22007.908536
4,1_1_2012-11-30,26128.508982


## Cell 7 — Log the Inference Run


In [7]:
with mlflow.start_run(run_name='Inference_Final_Submission'):
    mlflow.set_tags({
        'stage': 'inference',
        'registered_model': REGISTERED_MODEL_NAME,
        'model_version_or_stage': MODEL_STAGE_OR_VERSION,
        'source_run_id': final_model.metadata.run_id,
    })
    mlflow.log_metrics({
        'test_rows': len(test_raw),
        'predictions_generated': len(test_predictions),
        'negative_predictions_clipped': int(n_negative),
    })
    mlflow.log_artifact('submission_final.csv', artifact_path='submission')

print('Inference run logged.')

🏃 View run Inference_Final_Submission at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/0/runs/20c52359ef2b48789f99633170f5f5b5
🧪 View experiment at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/0
Inference run logged.
